In [1]:
# This code mounts Google Drive to the Colab environment and imports the main libraries that will be used throughout the project.
# Here, polars is used for data processing and analysis, pandas for additional table operations,plotly.express,
# and plotly.graph_objects for visualization, while Path is used for working with file paths.

from google.colab import drive
drive.mount('/content/drive')

import polars as pl
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

Mounted at /content/drive


In [2]:
# This code defines the folder path where the dataset files are stored and loads the movies, ratings, and tags tables from CSV files into Polars DataFrames.
# Here, Path is used to manage the file path more conveniently, while pl.read_csv() is used to read each file for analysis.

base = Path('/content/drive/MyDrive/Introduction2Python_projects/Polars_project')

movies  = pl.read_csv(base / "movies.csv", separator=",")
ratings = pl.read_csv(base / "ratings.csv", separator=",")
tags    = pl.read_csv(base / "tags.csv", separator=",")

The dataset used in this project represents a movie rating ecosystem and is composed of three related tables: **movies**, **ratings**, and **tags**. Each table describes a different part of the interaction between users and movies.

The **movies.csv** table contains information about individual movies, where each row corresponds to a unique movie identified by its movieId, along with its title and genres.

The **ratings.csv** table records the ratings that users assign to movies; each row represents a single rating event and includes the userId, movieId, rating score, and the timestamp indicating when the rating was given.

The **tags.csv** table stores user-generated tags for movies, where each row represents a tag added by a user to a specific movie, together with the corresponding userId, movieId, tag, and timestamp.

Overall, this dataset represents movies and user interactions with movies through ratings and tags, making it appropriate for data preprocessing, exploratory data analysis, and recommendation-oriented insights.

# Preprocess data

In [3]:
# This code displays the general structure of the movies table together with its main summary statistics.
# It helps provide an initial understanding of the columns, data types, and overall content of the dataset.

movies.describe()

statistic,movieId,title,genres
str,f64,str,str
"""count""",9742.0,"""9742""","""9742"""
"""null_count""",0.0,"""0""","""0"""
"""mean""",42200.353623,null,null
"""std""",52160.494854,null,null
"""min""",1.0,"""'71 (2014)""","""(no genres listed)"""
"""25%""",3248.0,null,null
"""50%""",7301.0,null,null
"""75%""",76251.0,null,null
"""max""",193609.0,"""À nous la liberté (Freedom for…","""Western"""


Based on this table, it appears that the movies dataset contains 9,742 unique movies, and there are no missing values in the main columns (movieId, title, genres). This shows that the movie information is complete and suitable for analysis. However, this table does not directly reflect user behavior, because it only provides information about the movies themselves. To better understand user behavior, the main focus should be on the ratings and tags tables, since they show how users interact with movies.

In [4]:
# This code generates the summary statistics for the ratings table and helps assess the overall structure of the data.
# It is especially useful for observing the distribution of the userId, movieId, rating, and timestamp columns and spotting possible anomalies.

ratings.describe()

statistic,userId,movieId,rating,timestamp
str,f64,f64,f64,f64
"""count""",100836.0,100836.0,100836.0,100836.0
"""null_count""",0.0,0.0,0.0,0.0
"""mean""",326.127564,19435.295718,3.501557,1.2059e9
"""std""",182.618491,35530.987199,1.042529,2.1626e8
"""min""",1.0,1.0,0.5,8.28124615e8
"""25%""",177.0,1199.0,3.0,1.0191e9
"""50%""",325.0,2991.0,3.5,1.1861e9
"""75%""",477.0,8121.0,4.0,1.4360e9
"""max""",610.0,193609.0,5.0,1.5378e9


Based on this table, it appears that the ratings dataset contains 100,836 rating records, and there are no missing values in any of the main columns (userId, movieId, rating, timestamp). The average rating is 3.5, which suggests that users generally give moderately positive ratings to movies. The ratings range from 0.5 to 5.0, showing that users use a wide scale when expressing their opinions. Also, since the dataset includes ratings from 610 users, it appears that the platform has a diverse set of user preferences and rating behaviors.

In [5]:
# This part shows the summary statistics of the tags table and allows an initial review of the tag data.
# It provides a first look at the column types as well as the general form of the tag and timestamp information.

tags.describe()

statistic,userId,movieId,tag,timestamp
str,f64,f64,str,f64
"""count""",3683.0,3683.0,"""3683""",3683.0
"""null_count""",0.0,0.0,"""0""",0.0
"""mean""",431.149335,27252.013576,null,1.3200e9
"""std""",158.472553,43490.558803,null,1.7210e8
"""min""",2.0,1.0,"""""artsy""""",1.1372e9
"""25%""",424.0,1263.0,null,1.1375e9
"""50%""",474.0,4454.0,null,1.2698e9
"""75%""",477.0,39292.0,null,1.4985e9
"""max""",610.0,193565.0,"""zombies""",1.5371e9


Based on this table, it appears that the tags dataset contains 3,683 tag records, and there are no missing values in the main columns (userId, movieId, tag, timestamp). This shows that the tagging data is complete and available for analysis. Compared to the ratings table, the number of tag records is much smaller, which suggests that users tag movies less frequently than they rate them. This may indicate that giving a rating is a more common and easier form of interaction, while adding tags requires more effort and is done by fewer users.

In [6]:
# This code displays the first few rows of the movies table.
# It makes it easier to directly observe the column names, the way the data is stored, and the overall appearance of the table.

movies.head()

movies.head()

movieId,title,genres
i64,str,str
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…"
2,"""Jumanji (1995)""","""Adventure|Children|Fantasy"""
3,"""Grumpier Old Men (1995)""","""Comedy|Romance"""
4,"""Waiting to Exhale (1995)""","""Comedy|Drama|Romance"""
5,"""Father of the Bride Part II (1…","""Comedy"""


In [7]:
# This part prints the first few rows of the ratings table.
# The aim is to inspect the structure of the rating data, the arrangement of the columns, and the format of the values at an initial stage.

ratings.head()

userId,movieId,rating,timestamp
i64,i64,f64,i64
1,1,4.0,964982703
1,3,4.0,964981247
1,6,4.0,964982224
1,47,5.0,964983815
1,50,5.0,964982931


In [8]:
# This code shows the first few rows of the tags table and provides an initial view of the tag data.
# It allows direct inspection of the user identifier, movie identifier, tag texts, and timestamp information.


tags.head()

userId,movieId,tag,timestamp
i64,i64,str,i64
2,60756,"""funny""",1445714994
2,60756,"""Highly quotable""",1445714996
2,60756,"""will ferrell""",1445714992
2,89774,"""Boxing story""",1445715207
2,89774,"""MMA""",1445715200


In [9]:
# This code shows the names of all columns in the movies table together with their data types.
# This view is useful for checking whether the data was read correctly and for planning the next preprocessing steps.


movies.schema

Schema([('movieId', Int64), ('title', String), ('genres', String)])

In [10]:
# The result shows that all columns are generally stored in the expected data types.

In [11]:
# This part displays the column names in the ratings table along with their data types.
# This step is especially important for checking whether the ID, rating, and timestamp columns have suitable types.


ratings.schema

Schema([('userId', Int64),
        ('movieId', Int64),
        ('rating', Float64),
        ('timestamp', Int64)])

In [12]:
# It appears that the column types in the tables are generally correct and the data structure matches the expected format.

In [13]:
# This code is used to inspect the column names in the tags table and their corresponding data types.
# Seeing how the text and timestamp values are read is important for the later cleaning stage.

tags.schema

Schema([('userId', Int64),
        ('movieId', Int64),
        ('tag', String),
        ('timestamp', Int64)])

In [14]:
# The result shows that all columns are generally stored in the expected data types.

In [15]:
# This code checks the number of missing values in each column of the movies table.
# Its purpose is to determine whether the dataset contains any missing values and to prepare for preprocessing.

movies.null_count()

movieId,title,genres
u32,u32,u32
0,0,0


In [16]:
# The result shows that there are no missing values in any column of the movies table.
# This means that there is no missing value issue in this table.

In [17]:
# This part checks the number of missing values in each column of the ratings table.
# It helps determine whether the data is complete and whether any values are missing.


ratings.null_count()

userId,movieId,rating,timestamp
u32,u32,u32,u32
0,0,0,0


In [18]:
# Based on the output, no missing values were detected in any column of the ratings table either.
# This means that the table can also be considered complete in terms of missing values.

In [19]:
# This code shows the number of missing values in each column of the tags table.
# This check is used to evaluate the completeness of the tag data and to guide the next cleaning steps.

tags.null_count()

userId,movieId,tag,timestamp
u32,u32,u32,u32
0,0,0,0


In [20]:
# The result shows that the number of missing values is also zero in all columns of the tags table.
# Therefore, since no missing values were found in any of the three tables, no additional filling step is required here.

In [21]:
# This code checks whether there are any fully duplicated rows in the movies table.
# The goal is to determine if the same record appears more than once and to evaluate the cleanliness of the data.

movies.is_duplicated().sum()

0

In [22]:
# The result shows that there are no fully duplicated rows in the movies table.
# This means that the table is clean in terms of duplicates.

In [23]:
# This part counts the number of completely identical rows in the ratings table.
# This check is carried out to see whether repeated records exist and to assess data quality before analysis.

ratings.is_duplicated().sum()

0

In [24]:
# Based on the output, there are no fully duplicated rows in the ratings table either.
# This means the rating data is clean in this respect and no additional removal step is necessary.

In [25]:
# This code identifies the number of fully duplicated rows in the tags table.
# Such a check is important for understanding whether the same data has been entered multiple times in exactly the same form.

tags.is_duplicated().sum()

0

In [26]:
# The result shows that there are no fully duplicated rows in the tags table either.
# Therefore, since no duplicate rows were found in any of the three tables, there are no extra records to remove at this stage.

## Data Analysis

### 1.1: The values in the rating column of the ratings table are checked by displaying the minimum and maximum values to confirm that they fall between 0.5 and 5.0.

In [27]:
ratings.select(pl.col("rating").min().alias("min_rating"),pl.col("rating").max().alias("max_rating"))

min_rating,max_rating
f64,f64
0.5,5.0


In [28]:
# The result shows that the minimum value in the rating column is 0.5, while the maximum value is 5.0.
# This indicates that the rating values are stored within the correct and sensible range.

### 1.2: The number of occurrences for each rating value is calculated and displayed in table format.

In [107]:
ratings_count_by_score = (ratings.group_by("rating").agg(pl.len().alias("count")).sort("count", descending=True))

In [108]:
ratings_count_by_score

rating,count
f64,u32
4.0,26818
3.0,20047
5.0,13211
3.5,13136
4.5,8551
2.0,7551
2.5,5550
1.0,2811
1.5,1791


In [31]:
# The result shows that 4.0 is the most frequently used rating, recorded 26,818 times.
# Values such as 3.0 and 5.0 also appear quite often, while lower ratings are given less frequently.
# This distribution suggests that users generally tend to assign medium to high scores.

### 1.3: The genres_list column is exploded, after which the data is grouped by genre and the average rating for each genre is calculated and displayed.

In [32]:
avg_rating_per_genre = (ratings.join(movies, on="movieId", how="inner").with_columns(pl.col("genres").str.split("|").alias("genre")).explode("genre").group_by("genre")
.agg(pl.col("rating").mean().alias("avg_rating")).sort("avg_rating", descending=True))

In [33]:
avg_rating_per_genre

genre,avg_rating
str,f64
"""Film-Noir""",3.920115
"""War""",3.808294
"""Documentary""",3.797785
"""Crime""",3.658294
"""Drama""",3.656184
…,…
"""Sci-Fi""",3.455721
"""Action""",3.447984
"""Children""",3.412956


In [34]:
# The result shows that Film-Noir has the highest average rating, with an average score of about 3.92.
# War and Documentary also appear among the more highly rated genres, while genres such as Horror and Comedy have relatively lower average ratings.
# Overall, these results indicate that user evaluations vary across genres.

### 1.4: The number of ratings given by each user is calculated, after which the minimum, maximum, and mean values of these counts are determined and displayed.

In [35]:
ratings_per_user = (ratings.group_by("userId").agg(pl.len().alias("ratings_count")))

In [36]:
ratings_per_user.select(pl.col("ratings_count").min().alias("min_ratings_per_user"),pl.col("ratings_count").max().alias("max_ratings_per_user"),
pl.col("ratings_count").mean().alias("mean_ratings_per_user"))

min_ratings_per_user,max_ratings_per_user,mean_ratings_per_user
u32,u32,f64
20,2698,165.304918


In [37]:
# The result shows that the minimum number of ratings given by a user is 20, while the maximum is 2698.
# On average, each user has given approximately 165.3 ratings.
# This difference indicates that users are not equally active on the platform, as some provide far more ratings than others.

### 1.5: The number of ratings given by each user is calculated, after which the users are sorted in descending order based on these counts and the top 5 most active users are displayed.

In [38]:
top_5_users = (ratings.group_by("userId").agg(pl.len().alias("ratings_c")).sort("ratings_c", descending=True).head(5))

In [39]:
top_5_users

userId,ratings_c
i64,u32
414,2698
599,2478
474,2108
448,1864
274,1346


In [40]:
# The result shows that user 414 gave the highest number of ratings, with a total of 2698 entries.
# It is followed by users 599, 474, 448, and 274, whose rating counts are also considerably higher than those of others.
# This result indicates that some users are highly active in the system.

### 2.1: The number of ratings and the average rating are first calculated for each movie. Then, the top 10 most frequently rated movies and the bottom 10 movies with the fewest ratings, provided that they have at least 5 ratings, are selected and their average ratings are compared.

In [41]:
movie_stats = (ratings.group_by("movieId").agg([pl.len().alias("rating_count"),pl.col("rating").mean().alias("avg_rating"),]).filter(pl.col("rating_count") >= 5))

In [42]:
movie_stats

movieId,rating_count,avg_rating
i64,u32,f64
4467,13,3.769231
105,23,3.282609
6932,7,3.5
103688,7,4.214286
3996,110,3.836364
…,…,…
2935,8,4.0
886,6,2.666667
27822,5,2.7


In [43]:
# The result shows that the rating_count and avg_rating were calculated for each movie, and only movies with at least 5 ratings were retained.
# The table presents movie-level summary statistics through the movieId, rating_count, and avg_rating columns.

In [44]:
top10 = movie_stats.sort("rating_count", descending=True).head(10)

In [45]:
top10

movieId,rating_count,avg_rating
i64,u32,f64
356,329,4.164134
318,317,4.429022
296,307,4.197068
593,279,4.16129
2571,278,4.192446
260,251,4.231076
480,238,3.75
110,237,4.031646
589,224,3.970982


In [46]:
# This result shows that the 10 most frequently rated movies have been selected and sorted in descending order by rating_count.
# Based on the displayed values, the most frequently rated movie in this group is movieId 356 with 329 ratings, followed by movieIds 318 and 296.
# At the same time, the average ratings of these movies range approximately from 3.75 to 4.43, which suggests that the most frequently rated movies are generally rated quite highly.

In [47]:
bottom10 = movie_stats.sort("rating_count").head(10)

In [48]:
bottom10

movieId,rating_count,avg_rating
i64,u32,f64
6724,5,3.6
7018,5,3.9
3169,5,3.3
7700,5,3.3
2769,5,3.4
4464,5,3.6
4231,5,3.5
5092,5,3.0
45928,5,3.7


In [49]:
# This result shows that the 10 least frequently rated movies have been selected, and each of them has the minimum threshold of 5 ratings.
# Based on the displayed values, the average ratings in this group range from 2.1 to 4.2, meaning that among low-rated-frequency movies there are both lower-rated and higher-rated examples.
# This also shows that average ratings for films with fewer ratings appear to be more variable.

### 2.2: The number of ratings received by each movie is calculated, and then the number of movies sharing the same rating count is determined and displayed in table form.

In [50]:
ratings_per_movie = (ratings.group_by("movieId").agg(pl.len().alias("rating_count")))

In [51]:
ratings_per_movie

movieId,rating_count
i64,u32
355,42
3142,4
6953,25
293,133
103980,4
…,…
71535,53
6791,5
118248,1


In [52]:
# The result shows that the number of ratings received by each movie has been calculated and displayed together with movieId.
# From the visible part, it can be observed that some movies received only 1 or 2 ratings, while others collected many more.
# This indicates that movies differ in terms of rating count and makes it possible to calculate the distribution of these counts in the next step.

In [53]:
distribution = (ratings_per_movie.group_by("rating_count").agg(pl.len().alias("num_movies")).sort("rating_count"))

In [54]:
distribution

rating_count,num_movies
u32,u32
1,3446
2,1298
3,800
4,530
5,382
…,…
278,1
279,1
307,1


In [55]:
# This result shows that most movies received only a small number of ratings.
# For example, 3,446 movies received only 1 rating, 1,298 movies received 2 ratings, and 800 movies received 3 ratings.
# At the same time, movies with very high rating_count values are quite rare.For instance, there is only 1 movie each with 278, 279, 307, 317, and 329 ratings.
# This distribution indicates that the dataset is dominated by movies with few ratings, while highly popular movies are relatively rare.

### 2.3: The number of ratings for each movie is calculated, then the movies are sorted in descending order based on this count and the top 20 most frequently rated movies are selected.

In [56]:
top20_most_rated_movies = (ratings.group_by("movieId").agg(pl.len().alias("rating_count")).join(movies, on="movieId", how="left").select(["movieId", "title", "rating_count"]).sort("rating_count", descending=True)
.head(20))

In [57]:
top20_most_rated_movies

movieId,title,rating_count
i64,str,u32
356,"""Forrest Gump (1994)""",329
318,"""Shawshank Redemption, The (199…",317
296,"""Pulp Fiction (1994)""",307
593,"""Silence of the Lambs, The (199…",279
2571,"""Matrix, The (1999)""",278
…,…,…
47,"""Seven (a.k.a. Se7en) (1995)""",203
780,"""Independence Day (a.k.a. ID4) …",202
150,"""Apollo 13 (1995)""",201


In [58]:
# The result shows that Forrest Gump (1994) is the most frequently rated movie, with 329 ratings.
# It is followed by films such as The Shawshank Redemption (1994), Pulp Fiction (1994), The Silence of the Lambs (1991), and The Matrix (1999), all of which also have very high rating counts.
# Overall, this list reflects the movies that attracted the most attention and received the highest number of ratings in the dataset.

### 2.4: The genres column is first split so that each genre is handled separately, and then the total number of unique genres is calculated. In addition, the frequency of all genres is determined, and the top 10 most common genres are displayed together with their counts.

In [59]:
genres_exploded = (movies.with_columns(pl.col("genres").str.split("|").alias("genre_list")).explode("genre_list"))

In [60]:
genres_exploded

movieId,title,genres,genre_list
i64,str,str,str
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Adventure"""
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Animation"""
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Children"""
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Comedy"""
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Fantasy"""
…,…,…,…
193583,"""No Game No Life: Zero (2017)""","""Animation|Comedy|Fantasy""","""Fantasy"""
193585,"""Flint (2017)""","""Drama""","""Drama"""
193587,"""Bungo Stray Dogs: Dead Apple (…","""Action|Animation""","""Action"""


In [61]:
# The result shows that multiple genres belonging to the same movie have been separated into individual rows.
# For example, for Toy Story (1995), the genres Adventure, Animation, Children, Comedy, and Fantasy appear separately.
# This makes it possible to count genres one by one and to calculate their frequencies correctly in the next step.

In [62]:
total_unique_genres = genres_exploded.select(pl.col("genre_list").n_unique().alias("total_unique_genres"))

In [63]:
total_unique_genres

total_unique_genres
u32
20


In [64]:
# The result shows that there are 20 unique genres in the dataset overall.
# This indicates that the movie data covers a diverse set of genre categories.

In [65]:
top10_genres = (genres_exploded.group_by("genre_list").agg(pl.len().alias("movie_count")).sort("movie_count", descending=True).head(10).rename({"genre_list": "genre"}))

In [66]:
top10_genres

genre,movie_count
str,u32
"""Drama""",4361
"""Comedy""",3756
"""Thriller""",1894
"""Action""",1828
"""Romance""",1596
"""Adventure""",1263
"""Crime""",1199
"""Sci-Fi""",980
"""Horror""",978


In [67]:
# The result shows that Drama is the most common genre in the dataset, appearing in 4361 movies.
# It is followed by Comedy with 3756 movies and Thriller with 1894 movies.
# The list indicates that Drama and Comedy are represented much more heavily than the other genres.

### 2.5: The number of movies and the average rating are calculated for each genre, and the result is presented in a sorted table.

In [68]:
genre_summary = (ratings.join(movies, on="movieId", how="inner").with_columns(pl.col("genres").str.split(by="|").alias("genre")).explode("genre")
.group_by("genre").agg([pl.col("movieId").n_unique().alias("n_movies"),pl.col("rating").mean().alias("avg_rating"),]).sort("avg_rating", descending=True))

In [69]:
genre_summary

genre,n_movies,avg_rating
str,u32,f64
"""Film-Noir""",85,3.920115
"""War""",381,3.808294
"""Documentary""",438,3.797785
"""Crime""",1196,3.658294
"""Drama""",4349,3.656184
…,…,…
"""Sci-Fi""",980,3.455721
"""Action""",1828,3.447984
"""Children""",664,3.412956


In [70]:
# The result shows that Film-Noir has the highest average rating, with about 3.92 across 85 movies.
# War and Documentary also appear near the top with high average ratings, whereas Horror and Comedy are seen near the lower end of the list.
# At the same time, the table also reveals differences in genre size.For example, Drama is represented by 4349 movies, while Film-Noir includes only 85 movies.

### 3.1: The number of tags assigned to each movie is calculated, after which the average, minimum, and maximum values of these counts are determined and displayed.


In [71]:
tags_per_movie = (tags.group_by("movieId").agg(pl.len().alias("tag_count")))

In [72]:
tags_per_movie

movieId,tag_count
i64,u32
5012,2
7646,1
6178,3
1984,1
2421,1
…,…
2572,1
2827,1
79132,26


In [73]:
# The result shows that the number of tags differs across movies.
# Some movies received only 1 tag, while others were assigned more tags.
# This table provides the basis for evaluating the overall level of tag usage across movies.

In [74]:
tag_stats = tags_per_movie.select(pl.col("tag_count").mean().alias("avg_tags_per_movie"),pl.col("tag_count").min().alias("min_tags_per_movie"),
pl.col("tag_count").max().alias("max_tags_per_movie"),)

In [75]:
tag_stats

avg_tags_per_movie,min_tags_per_movie,max_tags_per_movie
f64,u32,u32
2.342875,1,181


In [76]:
# The result shows that a movie receives about 2.34 tags on average.
# The minimum number of tags is 1, while the maximum number of tags reaches 181.
# This indicates that although many movies are tagged only a few times, some movies receive far more tags than others.

### 3.2: The number of times each tag has been applied is calculated, after which the tags are sorted in descending order by frequency and the top 20 most frequently used tags are displayed.

In [77]:
top20_tags = (tags.group_by("tag").agg(pl.len().alias("count")).sort("count", descending=True).head(20))

In [78]:
top20_tags

tag,count
str,u32
"""In Netflix queue""",131
"""atmospheric""",36
"""superhero""",24
"""thought-provoking""",24
"""Disney""",23
…,…
"""visually appealing""",19
"""politics""",18
"""music""",16


In [79]:
# The result shows that the most frequently used tag is "In Netflix queue", which was applied 131 times.
# It is followed by tags such as "atmospheric", "thought-provoking", "superhero", and "funny", although their counts are much lower compared to the first tag.
# This indicates that some tags are clearly more popular and more frequently used by users than others.

### 3.3: The proportion of ratings that are 4 or higher is calculated for each movie. Then, only movies with at least 5 ratings are kept, and the final table displays the movie ID, title, and the high_rating_proportion value.

In [80]:
high_rating_prop_by_movie = (ratings.group_by("movieId").agg([pl.len().alias("n_ratings"),(pl.col("rating") >= 4.0).mean().alias("high_rating_proportion")])
.filter(pl.col("n_ratings") >= 5)
.join(movies, on="movieId", how="left")
.select(["movieId", "title", "n_ratings", "high_rating_proportion"])
.sort("high_rating_proportion", descending=True))

In [81]:
high_rating_prop_by_movie

movieId,title,n_ratings,high_rating_proportion
i64,str,u32,f64
1192,"""Paris Is Burning (1990)""",5,1.0
177593,"""Three Billboards Outside Ebbin…",8,1.0
5008,"""Witness for the Prosecution (1…",9,1.0
942,"""Laura (1944)""",6,1.0
7983,"""Broadway Danny Rose (1984)""",5,1.0
…,…,…,…
67197,"""Knowing (2009)""",9,0.0
3394,"""Blind Date (1987)""",7,0.0
5094,"""Rollerball (2002)""",7,0.0


In [82]:
# The result shows that some movies have a high_rating_proportion of 1.0, meaning that all of their recorded ratings are considered high ratings.
# For example, in the displayed output, movies such as Swept Away, Broadway Danny Rose, The Conjuring, and Three Billboards Outside Ebbing, Missouri have a value of 1.0.
# On the other hand, movies like Jaws 3-D, Fantastic Four (2015), Eye of the Beholder, and 54 have a value of 0.0, which means that none of their ratings are 4 or higher.
# This indicates that user satisfaction is not the same across movies, as some are rated entirely positively while others receive no high ratings at all.

### 3.4: It is examined whether users have rated the same movie more than once. To do this, the data is first grouped by userId and movieId, then the proportion of repeated ratings is calculated for each user, and the top 10 users with the highest re-rating proportion are displayed.

In [83]:
user_movie_counts = (ratings.group_by(["userId", "movieId"]).agg(pl.len().alias("rating_count_per_movie")))

In [84]:
user_movie_counts

userId,movieId,rating_count_per_movie
i64,i64,u32
256,3717,1
157,3268,1
458,208,1
516,4333,1
140,2861,1
…,…,…
68,3623,1
596,89745,1
524,788,1


In [85]:
# The result shows that a new measure called rating_count_per_movie has been calculated for each userId and movieId pair.
# This table shows how many times the same user rated the same movie and forms the basis for calculating the re-rating proportion.

In [86]:
user_movie_counts = user_movie_counts.with_columns(pl.when(pl.col("rating_count_per_movie") > 1).then(pl.col("rating_count_per_movie") - 1).otherwise(0).alias("re_rating_count"))

In [87]:
user_movie_counts

userId,movieId,rating_count_per_movie,re_rating_count
i64,i64,u32,u32
256,3717,1,0
157,3268,1,0
458,208,1,0
516,4333,1,0
140,2861,1,0
…,…,…,…
68,3623,1,0
596,89745,1,0
524,788,1,0


In [88]:
# The result shows that all rows in the displayed output have a re_rating_count value of 0.
# This means that, in the shown sample, there are no additional or repeated ratings from the same user for the same movie.
# Therefore, this column is used in the next step to calculate the overall re-rating proportion for each user.

In [89]:
user_rerating_stats = (user_movie_counts.group_by("userId").agg([pl.col("rating_count_per_movie").sum().alias("total_ratings"),
pl.col("re_rating_count").sum().alias("total_re_ratings")]).with_columns((pl.col("total_re_ratings") / pl.col("total_ratings")).alias("re_rating_proportion"))
.sort("re_rating_proportion", descending=True))

In [90]:
user_rerating_stats

userId,total_ratings,total_re_ratings,re_rating_proportion
i64,u32,u32,f64
165,65,0,0.0
376,133,0,0.0
123,56,0,0.0
266,180,0,0.0
579,73,0,0.0
…,…,…,…
228,25,0,0.0
360,25,0,0.0
398,46,0,0.0


In [91]:
# The result shows that the total_ratings, total_re_ratings, and re_rating_proportion values have been calculated for each user.
# In the displayed portion, the re_rating_proportion values appear as 0.0, meaning that no repeated rating share is observed in that sample.
# This table provides the overall user-level statistics needed to evaluate re-rating behavior.

In [92]:
top_10_users = user_rerating_stats.head(10)

In [93]:
top_10_users

userId,total_ratings,total_re_ratings,re_rating_proportion
i64,u32,u32,f64
165,65,0,0.0
376,133,0,0.0
123,56,0,0.0
266,180,0,0.0
579,73,0,0.0
442,20,0,0.0
281,21,0,0.0
3,39,0,0.0
554,89,0,0.0


In [94]:
# The result shows that all of the selected top 10 users have a re_rating_proportion value of 0.0.
# This indicates that no repeated rating share is observed for the users shown in this output.
# At the same time, the table shows that although users differ in their total number of ratings, the total_re_ratings column remains 0 for all of them.

### 3.5: The average rating and the number of ratings are calculated for each movie. Then, only movies with at least 10 ratings are retained, the result is joined with the movies table to add movie titles, and the top 20 movies with the highest average ratings are displayed.

In [95]:
top_20_movies = (ratings.group_by("movieId").agg([pl.len().alias("rating_count"),pl.col("rating").mean().alias("average_rating")]).filter(pl.col("rating_count") >= 10)
.join(movies.select(["movieId", "title"]), on="movieId", how="left").select(["movieId", "title", "average_rating", "rating_count"]).sort("average_rating", descending=True)
.head(20))

In [96]:
top_20_movies

movieId,title,average_rating,rating_count
i64,str,f64,u32
1041,"""Secrets & Lies (1996)""",4.590909,11
3451,"""Guess Who's Coming to Dinner (…",4.545455,11
1178,"""Paths of Glory (1957)""",4.541667,12
1104,"""Streetcar Named Desire, A (195…",4.475,20
2360,"""Celebration, The (Festen) (199…",4.458333,12
…,…,…,…
7156,"""Fog of War: Eleven Lessons fro…",4.307692,13
1209,"""Once Upon a Time in the West (…",4.305556,18
1204,"""Lawrence of Arabia (1962)""",4.3,45


In [97]:
# The result shows that among the movies with the highest average ratings are Secrets & Lies (1996), Guess Who's Coming to Dinner, Paths of Glory, and A Streetcar Named Desire.
# In the displayed output, the highest average_rating is around 4.59, and since all movies in the list have at least 10 ratings, the result can be considered more reliable.
# This suggests that these movies are generally rated very highly by users.

### 3.6: The number of ratings and the average rating are calculated for each movie, and then the correlation between these two measures is determined. This makes it possible to evaluate whether there is a linear relationship between how many ratings a movie receives and its average rating.

In [98]:
movie_stats = (ratings.group_by("movieId").agg([pl.len().alias("rating_count"),pl.col("rating").mean().alias("average_rating")]))

In [99]:
movie_stats

movieId,rating_count,average_rating
i64,u32,f64
39801,2,2.75
153,137,2.916058
4738,4,4.0
118530,1,4.5
65037,1,3.5
…,…,…
85399,1,1.0
40491,1,5.0
6722,1,2.5


In [100]:
# The result shows that the rating_count and average_rating values have been calculated separately for each movie.
# From the displayed part, it is clear that some movies received only a few ratings, while others have much higher rating_count values, and the average ratings also vary across movies.
# This table serves as the basis for evaluating the relationship between the number of ratings a movie receives and its average rating.

In [101]:
correlation_result = movie_stats.select(pl.corr("rating_count", "average_rating").alias("correlation")).item()

In [102]:
correlation_result

0.12725857359560638

In [103]:
# The result shows that the correlation is approximately 0.127.
# This value indicates a weak positive relationship, meaning that movies with more ratings tend to have slightly higher average ratings, but the relationship is not strong.
# Overall, receiving many ratings does not necessarily mean that a movie will have a high average rating.

In [104]:
fig = px.scatter(movie_stats.to_pandas(),x="rating_count",y="average_rating",title=f"Relationship Between Number of Ratings and Average Rating (corr = {correlation_result:.3f})",
labels={"rating_count": "Number of Ratings","average_rating": "Average Rating"})
fig.show()

In [105]:
# The scatter plot shows that the points do not align closely along a clear line.
# This is consistent with the previously calculated correlation value of 0.127, indicating a weak positive relationship.
# In other words, movies with more ratings may have slightly higher average ratings, but the trend is neither strong nor very clear.

**Conclusion**

This project analyzed the movies, ratings, and tags tables to understand both the structure of the dataset and user behavior. The results show that the dataset is complete and suitable for analysis, since there are no missing values in the main columns of the three tables. The findings also indicate that user interaction is reflected much more strongly in the ratings table than in the tags table, because the number of ratings is substantially higher than the number of tags.

The rating analysis shows that the average rating is **3.5**, while the most frequently given rating is **4.0**, suggesting that users generally tend to assign medium to high scores. At the same time, user activity is not evenly distributed, as some users contribute far more ratings than others. At the movie level, most films receive only a small number of ratings, whereas only a limited number of movies attract very high attention. In genre analysis, Drama appears as the most common genre in the dataset, while Film-Noir has the highest average rating. Overall, these findings suggest that ratings provide the strongest signal for understanding user preferences, while tags and genres provide valuable supporting context.

**Business Insights**

From a business perspective, the results suggest that recommendation systems should prioritize movies that have both high average ratings and high rating counts, since these films reflect both user satisfaction and popularity. At the same time, highly rated but less frequently rated movies can be recommended to users who are interested in more selective or niche content.

In addition, genre and tag information can improve personalization. Genre patterns help identify which categories are most common and which ones receive stronger audience appreciation, while frequently used tags reveal how users describe movies in terms of mood, theme, or style. Since the relationship between rating count and average rating is weak, popularity alone should not be used as the main recommendation criterion. A more effective strategy would be to combine rating score, rating count, genre, and tag information to generate more accurate and balanced movie recommendations.